In [0]:
passengers_day1 = [
    (101, 'Rahul Sharma', 'Hyderabad', 'Economy', 'India'),
    (102, 'Priya Reddy', 'Bangalore', 'Business', 'India'),
    (103, 'Amit Kumar', 'Mumbai', 'Economy', 'India'),
    (104, 'Sneha Patel', 'Delhi', 'Premium Economy', 'India'),
    (105, 'Farhan Ali', 'Chennai', 'Economy', 'India')
]
columns = ['passenger_id', 'passenger_name', 'city', 'travel_class', 'country']
df_day1 = spark.createDataFrame(passengers_day1, columns)

passengers_day2 = [
    (102, 'Priya Reddy', 'Bangalore', 'First Class', 'India'),
    (104, 'Sneha Patel', 'Hyderabad', 'Premium Economy', 'India'),
    (106, 'Neha Singh', 'Pune', 'Economy', 'India'),
    (107, 'Arjun Verma', 'Kochi', 'Business', 'India')
]
df_day2 = spark.createDataFrame(passengers_day2, columns)
print('Datasets created successfully')

Datasets created successfully


In [0]:
#1
spark.sql('DROP TABLE IF EXISTS passengers_delta')
df_day1.write.format('delta').mode('overwrite').saveAsTable('passengers_delta')
print('passengers_delta table created')

passengers_delta table created


In [0]:
#2
df_day1.write.format('delta').mode('overwrite').saveAsTable('passengers_delta')
print('Day1 data loaded into passengers_delta')

Day1 data loaded into passengers_delta


In [0]:
#3
count = spark.read.format('delta').table('passengers_delta').count()
print('Record count:', count)

Record count: 5


In [0]:
#4
spark.read.format('delta').table('passengers_delta').show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
#5
spark.sql('DESCRIBE HISTORY passengers_delta').show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+----------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                                               |operation                        |operationParameters                                                                                                                                                                

In [0]:
#6
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, 'passengers_delta')
delta_table.alias('target').merge(
    df_day2.alias('source'),
    'target.passenger_id = source.passenger_id'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print('Merge completed')

Merge completed


In [0]:
#7
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, 'passengers_delta')
delta_table.alias('target').merge(
    df_day2.alias('source'),
    'target.passenger_id = source.passenger_id'
).whenMatchedUpdateAll().execute()
print('Existing passengers updated')

Existing passengers updated


In [0]:
#8
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, 'passengers_delta')
delta_table.alias('target').merge(
    df_day2.alias('source'),
    'target.passenger_id = source.passenger_id'
).whenNotMatchedInsertAll().execute()
print('New passengers inserted')

New passengers inserted


In [0]:
#9
spark.sql("SELECT passenger_id, passenger_name, travel_class FROM passengers_delta WHERE passenger_id = 102").show()

+------------+--------------+------------+
|passenger_id|passenger_name|travel_class|
+------------+--------------+------------+
|         102|   Priya Reddy| First Class|
+------------+--------------+------------+



In [0]:
#10
spark.sql("SELECT * FROM passengers_delta WHERE passenger_id = 106").show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
#11
spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta').show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
#12
spark.read.format('delta').table('passengers_delta').show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
#13
df_v0 = spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta')
df_latest = spark.read.format('delta').table('passengers_delta')
print('Version 0:')
df_v0.show()
print('Latest Version:')
df_latest.show()

Version 0:
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+

Latest Version:
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         10

In [0]:
#14
print('Passenger 102 - Version 0:')
spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta') \
    .filter('passenger_id = 102').show()
print('Passenger 102 - Latest:')
spark.read.format('delta').table('passengers_delta') \
    .filter('passenger_id = 102').show()

Passenger 102 - Version 0:
+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+

Passenger 102 - Latest:
+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
#15
print('Passenger 104 - Version 0:')
spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta') \
    .filter('passenger_id = 104').show()
print('Passenger 104 - Latest:')
spark.read.format('delta').table('passengers_delta') \
    .filter('passenger_id = 104').show()

Passenger 104 - Version 0:
+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+

Passenger 104 - Latest:
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
#16
spark.sql('OPTIMIZE passengers_delta')
print('OPTIMIZE completed')

OPTIMIZE completed


In [0]:
#17
spark.sql('OPTIMIZE passengers_delta ZORDER BY (city)')
print('ZORDER BY city completed')

ZORDER BY city completed


In [0]:
#18
spark.sql('DELETE FROM passengers_delta WHERE passenger_id = 105')
print('Passenger 105 deleted')
spark.sql('SELECT * FROM passengers_delta').show()

Passenger 105 deleted
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
#19
spark.sql('DESCRIBE HISTORY passengers_delta').show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+----------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
#20
spark.sql('VACUUM passengers_delta RETAIN 0 HOURS')
print('VACUUM completed')

---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-4695000899236472>, line 2
      1 #20
----> 2 spark.sql('VACUUM passengers_delta RETAIN 0 HOURS')
      3 print('VACUUM completed')

File /databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/session.py:653, in SparkSession.sql(self, sqlQuery, args)
    651 def sql(self, sqlQuery: str, args: Optional[Union[Dict[str, Any], List]] = None) -> "DataFrame":
    652     cmd = SQL(sqlQuery, args)
--> 653     data, properties = self.client.execute_command(cmd.command(self._client))
    654     if "sql_command_result" in properties:
    655         return DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py:1208, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1206     req